# Laboratorium 1: od problemu biznesowego do projektu Data Mining

**Przypadek:** NovaTel — fikcyjny operator usług abonamentowych.  
**Cel:** zrozumieć problem biznesowy, rozpoznać dane i ich jakość oraz zaplanować projekt według CRISP-DM.

> Na tym laboratorium nie budujemy modelu. Model bez poprawnie zdefiniowanego problemu jest zwykle bardzo sprawnym sposobem uzyskania precyzyjnej odpowiedzi na niewłaściwe pytanie.

## Zasady pracy

1. Każdy wniosek powinien wynikać z danych albo z opisu sytuacji biznesowej.
2. Nie poprawiajcie jeszcze danych — najpierw je zdiagnozujcie.

In [29]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
DATA_PATH = Path("dane_novatel_lab1.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/lab1_data_mining_materialy/dane_novatel_lab1.csv")

df = pd.read_csv(DATA_PATH)

## Zadanie 1. Pierwsze spojrzenie na dane

Sprawdź wymiary zbioru, pięć pierwszych rekordów i nazwy kolumn.

In [30]:
print("Wymiary:", df.shape)
display (df.head(5))
print("Kolumny: ",df.columns)

Wymiary: (325, 21)


,customer_id,snapshot_date,wiek,płeć,województwo,wielkość_miejscowości,typ_umowy,staż_miesiące,opłata_miesięczna_pln,średnie_użycie_h_mies,...,kontakty_z_obsługą_3m,reklamacje_6m,satysfakcja_1_5,zgoda_marketingowa,kanał_pozyskania,rabat_proc,dni_od_ostatniego_logowania,dochód_miesięczny_pln,odejście_90d,data_zamknięcia_konta
0,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04
1,NT10110,2026-10-01,45.0,inna/nie podano,Mazowieckie,średnie miasto,12 miesięcy,67,100.62,63.9,...,1,3,4.0,nie,salon,13.0,9,18840.0,nie,NaN
2,NT10248,2026-10-01,36.0,M,Pomorskie,małe miasto,12 miesięcy,16,115.73,28.7,...,2,0,3.0,tak,salon,7.0,6,4220.0,nie,NaN
3,NT10009,2026-10-01,34.0,M,Dolnośląskie,średnie miasto,12 miesięcy,10,110.10,39.0,...,1,0,3.0,nie,telefon,4.0,4,9020.0,nie,NaN
4,NT10093,2026-10-01,34.0,M,Mazowieckie,duże miasto,miesięczna,32,117.50,38.8,...,1,0,4.0,nie,online,15.0,5,10260.0,tak,2026-12-26


Kolumny:  Index(['customer_id', 'snapshot_date', 'wiek', 'płeć', 'województwo',
       'wielkość_miejscowości', 'typ_umowy', 'staż_miesiące',
       'opłata_miesięczna_pln', 'średnie_użycie_h_mies',
       'opóźnienia_płatności_6m', 'kontakty_z_obsługą_3m', 'reklamacje_6m',
       'satysfakcja_1_5', 'zgoda_marketingowa', 'kanał_pozyskania',
       'rabat_proc', 'dni_od_ostatniego_logowania', 'dochód_miesięczny_pln',
       'odejście_90d', 'data_zamknięcia_konta'],
      dtype='str')


**Do karty pracy:**  
- Ile jest obserwacji i zmiennych?  
- Co jest jednostką obserwacji?  
- Czy każda kolumna jest potencjalnie przydatna przed podjęciem decyzji?

## Zadanie 2. Typy danych i zmienna celu

In [37]:
df.info()
counts = df['odejście_90d'].value_counts(dropna=False)
share = df['odejście_90d'].value_counts(dropna=False, normalize=True).mul(100).round(1).astype(str) + '%'
print("Wartości zmiennej celu: ")
print(counts)
print("Udział proporcjonalny: ")
print(share)

<class 'pandas.DataFrame'>
RangeIndex: 325 entries, 0 to 324
Data columns (total 21 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  325 non-null    str    
 1   snapshot_date                325 non-null    str    
 2   wiek                         313 non-null    float64
 3   płeć                         325 non-null    str    
 4   województwo                  325 non-null    str    
 5   wielkość_miejscowości        325 non-null    str    
 6   typ_umowy                    325 non-null    str    
 7   staż_miesiące                325 non-null    int64  
 8   opłata_miesięczna_pln        325 non-null    float64
 9   średnie_użycie_h_mies        325 non-null    float64
 10  opóźnienia_płatności_6m      325 non-null    int64  
 11  kontakty_z_obsługą_3m        325 non-null    int64  
 12  reklamacje_6m                325 non-null    int64  
 13  satysfakcja_1_5              30

**Do karty pracy:**  
- Jaki jest udział klientów, którzy odeszli?  
- Czy klasy są zrównoważone?  
- Dlaczego sama accuracy mogłaby później wprowadzać w błąd?

## Zadanie 3. Braki danych

In [43]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("Liczba braków"))

,Liczba braków
data_zamknięcia_konta,296
dochód_miesięczny_pln,28
satysfakcja_1_5,19
wiek,12
kanał_pozyskania,7


**Do karty pracy:**  
Wskaż kolumny z brakami i zaproponuj pytania, które należałoby zadać właścicielowi danych przed ich uzupełnianiem lub usuwaniem.

## Zadanie 4. Duplikaty i identyfikatory

In [52]:
print("Lizba duplikatów: ",df.duplicated().sum())
print("Lizba zduplikowanych id: ", df['customer_id'].duplicated().sum())
display(df[df['customer_id'].duplicated(keep=False)].sort_values('customer_id'))


Lizba duplikatów:  5
Lizba zduplikowanych id:  5


,customer_id,snapshot_date,wiek,płeć,województwo,wielkość_miejscowości,typ_umowy,staż_miesiące,opłata_miesięczna_pln,średnie_użycie_h_mies,...,kontakty_z_obsługą_3m,reklamacje_6m,satysfakcja_1_5,zgoda_marketingowa,kanał_pozyskania,rabat_proc,dni_od_ostatniego_logowania,dochód_miesięczny_pln,odejście_90d,data_zamknięcia_konta
22,NT10046,2026-10-01,38.0,K,Dolnośląskie,średnie miasto,miesięczna,34,51.95,10.7,...,1,2,2.0,nie,salon,4.0,2,9040.0,nie,NaN
60,NT10046,2026-10-01,38.0,K,Dolnośląskie,średnie miasto,miesięczna,34,51.95,10.7,...,1,2,2.0,nie,salon,4.0,2,9040.0,nie,NaN
192,NT10142,2026-10-01,32.0,K,Wielkopolskie,duże miasto,miesięczna,39,57.25,45.3,...,1,0,NaN,tak,online,20.0,4,5960.0,nie,NaN
318,NT10142,2026-10-01,32.0,K,Wielkopolskie,duże miasto,miesięczna,39,57.25,45.3,...,1,0,NaN,tak,online,20.0,4,5960.0,nie,NaN
14,NT10191,2026-10-01,64.0,M,Wielkopolskie,średnie miasto,12 miesięcy,26,39.18,16.7,...,3,0,3.0,tak,partner,14.0,2,9490.0,nie,NaN
307,NT10191,2026-10-01,64.0,M,Wielkopolskie,średnie miasto,12 miesięcy,26,39.18,16.7,...,3,0,3.0,tak,partner,14.0,2,9490.0,nie,NaN
228,NT10216,2026-10-01,50.0,K,Dolnośląskie,duże miasto,24 miesiące,41,92.65,48.0,...,2,0,5.0,tak,telefon,13.0,23,4630.0,nie,NaN
259,NT10216,2026-10-01,50.0,K,Dolnośląskie,duże miasto,24 miesiące,41,92.65,48.0,...,2,0,5.0,tak,telefon,13.0,23,4630.0,nie,NaN
0,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04
297,NT10234,2026-10-01,38.0,M,Mazowieckie,wieś,miesięczna,9,96.23,32.3,...,1,0,4.0,tak,online,24.0,3,13700.0,tak,2026-12-04


## Zadanie 5. Kategorie niespójne

In [56]:
for col in ["województwo","kanał_pozyskania","typ_umowy"]:
    print("\n",col)
    print(sorted(df[col].dropna().unique()))


 województwo
['Dolnośląskie', 'MAZOWIECKIE', 'Mazowieckie', 'Małopolskie', 'Pomorskie', 'Pomorskie ', 'Wielkopolskie', 'dolnośląskie', 'Śląskie']

 kanał_pozyskania
['Online', 'SALON', 'online', 'partner', 'salon', 'telefon', 'telefon ']

 typ_umowy
['12 miesięcy', '24 miesiące', 'miesięczna']


## Zadanie 6. Wartości podejrzane

In [79]:
display(df[["wiek","staż_miesiące","opłata_miesięczna_pln","satysfakcja_1_5"]].describe())
invalid = pd.concat([
    df[(df["wiek"] < 18) | (df["wiek"] > 100)].assign(problem="wiek poza zakresem"),
    df[df["opłata_miesięczna_pln"] < 0].assign(problem="opłata ujemna"),
    df[(df["satysfakcja_1_5"] < 1) | (df["satysfakcja_1_5"] > 5)].assign(problem="zła satysfakcja"),
    df[df["staż_miesiące"] <= 0].assign(problem="zły staż")])

display(invalid[["customer_id","wiek","staż_miesiące","opłata_miesięczna_pln","satysfakcja_1_5","problem"]])


# Następnie wyszukaj rekordy:
# - z wiekiem poza rozsądnym zakresem,
# - z opłatą miesięczną poniżej zera,
# - z satysfakcją poza skalą 1–5,
# - ze stażem równym 0 lub mniejszym.


,wiek,staż_miesiące,opłata_miesięczna_pln,satysfakcja_1_5
count,313.000000,325.000000,325.000000,306.000000
mean,40.798722,27.966154,95.652985,3.673203
std,13.187543,17.492558,28.663192,0.977317
min,18.000000,0.000000,-49.990000,1.000000
25%,32.000000,15.000000,76.860000,3.000000
50%,40.000000,25.000000,95.080000,4.000000
75%,48.000000,39.000000,115.100000,4.000000
max,129.000000,88.000000,175.860000,7.000000


,customer_id,wiek,staż_miesiące,opłata_miesięczna_pln,satysfakcja_1_5,problem
16,NT10146,129.0,3,100.81,5.0,wiek poza zakresem
202,NT10089,47.0,39,-49.99,4.0,opłata ujemna
264,NT10296,18.0,6,145.11,7.0,zła satysfakcja
135,NT10260,47.0,0,124.38,4.0,zły staż


## Zadanie 7. Prosta eksploracja biznesowa

**Do karty pracy:**  
Zapisz maksymalnie trzy obserwacje. To mają być hipotezy lub sygnały do dalszej analizy, a nie ostateczne dowody przyczynowości.

## Zadanie 8. CRISP-DM

Uzupełnij w karcie pracy plan projektu dla sześciu etapów:

1. Business Understanding  
2. Data Understanding  
3. Data Preparation  
4. Modeling  
5. Evaluation  
6. Deployment

Dla każdego etapu podaj:
- najważniejsze pytanie,
- wymagany rezultat,
- ryzyko lub warunek powodzenia.

## Zadanie 9. Etyka, prywatność i data leakage

In [ ]:
#jedna kolumna ktora zawiera problem

Odpowiedz:

- Czy `data_zamknięcia_konta` może być używana do przewidywania odejścia przed jego wystąpieniem?
- Jakie ryzyka wiążą się z wykorzystaniem wieku, płci, miejsca zamieszkania i dochodu?
- Czy NovaTel powinien automatycznie odmawiać ofert lub podnosić ceny na podstawie modelu?
- Jakie dane naprawdę są konieczne do rozwiązania problemu?

## Oddanie

Oddaj:
- uzupełnioną kartę pracy,
- notebook z wykonanymi analizami,
- jednozdaniową rekomendację dalszego działania dla NovaTel.